# ShelfRank: select ESCI seed queries

CPU only, free Colab is enough. Upload this one file, choose **Runtime > Run all**, and save the three
files it downloads into `data/esci/` in the repository.

It reads the ESCI US training queries with their Exact products, keeps queries whose products look like
home goods (plus some that do not), removes anything that resembles a WANDS query, and writes query
strings only. The code cells below are copies of the repository's tested modules.

In [ ]:
# Every path and setting is here. Nothing below needs editing.
import os, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
DEFAULT_WORK = "/content/shelfrank_seed" if IN_COLAB else (
    "/kaggle/working/shelfrank_seed" if Path("/kaggle").exists() else "shelfrank_seed")
WORK_DIR = Path(os.environ.get("SHELFRANK_WORK", DEFAULT_WORK))
EXAMPLES = WORK_DIR / "esci" / "examples.parquet"
PRODUCTS = WORK_DIR / "esci" / "products.parquet"
WANDS_QUERIES = WORK_DIR / "wands" / "query.csv"
OUT_DIR = WORK_DIR / "out"
DOWNLOAD = os.environ.get("SHELFRANK_DOWNLOAD", "1") == "1"

SETTINGS = {'locale': 'us', 'split': 'train', 'n_seed': 2000, 'n_dev': 100, 'home_share': 0.85, 'min_votes': 2, 'min_share': 0.4, 'near_duplicate_jaccard': 0.75, 'seed': 20260}
SOURCES = {
 "examples": {
  "url": "https://media.githubusercontent.com/media/amazon-science/esci-data/7916cdf6ab75a462e77f20ab40428a10923998d5/shopping_queries_dataset/shopping_queries_dataset_examples.parquet",
  "sha256": "4a735b693b4a424a6fc67f5be6e4c811495c488bbf66d02a602d308b2744263a"
 },
 "products": {
  "url": "https://media.githubusercontent.com/media/amazon-science/esci-data/7916cdf6ab75a462e77f20ab40428a10923998d5/shopping_queries_dataset/shopping_queries_dataset_products.parquet",
  "sha256": "25124442d064d64b26f74082d6fa09438d679efc0c183cf28d19064a2b65a265"
 },
 "wands_queries": {
  "url": "https://raw.githubusercontent.com/wayfair/WANDS/3b74dcf4ba29ab8ff3e6a50b5b09fc627cb882b5/dataset/query.csv",
  "sha256": "63b61660560fecc33ec490804c7e2b81402ee3e7c31a9cbb5e03736639f68e95"
 }
}
CODE_SHA256 = {
 "shelfrank/text/normalize.py": "547f5fc5d8702035166a0ec1a72c2018bd5a73f66371356f52ac6d9fd15045ce",
 "shelfrank/text/near_duplicates.py": "327589be5d32e67407a43d699c6b1d0d2dfc549ed2f64c49d70e1644ea33ff16",
 "shelfrank/data/download.py": "c5208ef25abede79441a0992a46bbbc1c77523f7dcf2f2b88499e0b134dd37ff",
 "shelfrank/qu/seed_selection.py": "45c92fb25402f6778f3d0c78f1f5a34c1a456ce5bd47c5ab3aee9f159d63103d",
 "data/class_terms.json": "e37028f9a7d1f71fd3b07418c531273207000735c52dda27d4ad934ed6859c1a"
}

for folder in ("shelfrank/text", "shelfrank/data", "shelfrank/qu", "data"):
    (WORK_DIR / folder).mkdir(parents=True, exist_ok=True)
for package in ("shelfrank", "shelfrank/text", "shelfrank/data", "shelfrank/qu"):
    (WORK_DIR / package / "__init__.py").touch()
os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR.resolve()))
started = time.time()
print("working in", WORK_DIR.resolve())

In [ ]:
%%writefile shelfrank/text/normalize.py
"""One text pipeline for queries and products, so that both sides always agree.

Every rule is a general property of English or of catalogue typography. None was derived from WANDS
queries.
"""

import re
import unicodedata

_PUNCTUATION = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"', "″": '"', "×": "x", "–": "-"})
# A quote mark or "in" counts as inches only straight after a number, since "in" alone is a
# preposition, and not before another number, since "3 in 1" is not a length.
_INCH = re.compile(r"(?<=\d)\s*(?:''|\"|inch(?:es)?\b|in\b(?!\s*\d)\.?)")
_FOOT = re.compile(r"(?<=\d)\s*(?:'(?!')|feet\b|foot\b|ft\b\.?)")
_POSSESSIVE = re.compile(r"(?<=[a-z])\s*'s\b")
_TOKEN = re.compile(r"\d+(?:\.\d+)?|[a-z]+")

STOPWORDS = frozenset({"a", "an", "and", "the", "of", "for", "with", "in", "on", "to", "by", "or"})

# British to American, limited to words that occur in home goods.
SPELLING = {
    "aluminium": "aluminum",
    "armour": "armor",
    "centre": "center",
    "centrepiece": "centerpiece",
    "colour": "color",
    "coloured": "colored",
    "fibre": "fiber",
    "grey": "gray",
    "jewellery": "jewelry",
    "metre": "meter",
    "mould": "mold",
    "moulding": "molding",
    "organiser": "organizer",
    "plough": "plow",
    "storey": "story",
}
IRREGULAR_PLURALS = {
    "children": "child",
    "knives": "knife",
    "leaves": "leaf",
    "loaves": "loaf",
    "men": "man",
    "mice": "mouse",
    "shelves": "shelf",
    "wolves": "wolf",
    "women": "woman",
}
# Singular already, although they end in a plural-looking s.
INVARIANT = frozenset({"bus", "canvas", "christmas", "gas", "lens", "plus", "series", "stainless"})


def normalize(text: str) -> str:
    """Lowercase, strip accents, unify quotes and spell out inch and foot marks."""
    folded = unicodedata.normalize("NFKD", text.translate(_PUNCTUATION).lower())
    plain = "".join(char for char in folded if not unicodedata.combining(char))
    plain = _POSSESSIVE.sub("", plain)
    plain = _INCH.sub(" inch ", plain)
    plain = _FOOT.sub(" ft ", plain)
    return re.sub(r"\s+", " ", plain).strip()


def singular(token: str) -> str:
    """Strip a regular plural ending.

    A conservative rule rather than a Porter stemmer: conflating "curtained" with "curtain" or
    "bedding" with "bed" would merge different product types.
    """
    if token in IRREGULAR_PLURALS:
        return IRREGULAR_PLURALS[token]
    if token in INVARIANT or len(token) <= 3 or not token.endswith("s"):
        return token
    if token.endswith(("ss", "us", "is")):
        return token
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith(("sses", "shes", "ches", "xes", "zes")):
        return token[:-2]
    return token[:-1]


def tokenize(text: object) -> list[str]:
    """Split into index terms; non-text (a missing cell) gives no terms.

    Hyphens split, so "mid-century" matches "mid century". Digits and letters split, so "54in" and
    "54 in" agree.
    """
    if not isinstance(text, str):
        return []
    tokens = []
    for raw in _TOKEN.findall(normalize(text)):
        token = singular(SPELLING.get(raw, raw))
        token = SPELLING.get(token, token)
        if token not in STOPWORDS:
            tokens.append(token)
    return tokens


In [ ]:
%%writefile shelfrank/text/near_duplicates.py
"""Find queries similar enough that putting them in different folds would leak labels."""

import re
from collections.abc import Mapping
from itertools import combinations

from shelfrank.text.normalize import tokenize

_TOKEN = re.compile(r"[a-z0-9]+")


def token_key(query: str) -> frozenset[str]:
    """Reduce a query to a word-order-free set with plural endings removed.

    Deliberately crude: this only has to over-group near-identical queries, not rank anything.
    """
    tokens = _TOKEN.findall(query.lower())
    return frozenset(t[:-1] if len(t) > 3 and t.endswith("s") else t for t in tokens)


def leakage_keys(query: str) -> tuple[frozenset[str], ...]:
    """Every key under which two queries may count as the same.

    The frozen folds use ``token_key``, which misses "vanity" against "vanities". Changing it would
    change the folds, so leakage checks add the full normaliser. No WANDS pair differs between them.
    """
    return tuple(dict.fromkeys(k for k in (token_key(query), frozenset(tokenize(query))) if k))


def jaccard(a: frozenset[str], b: frozenset[str]) -> float:
    """Set overlap; two empty sets are unrelated so that blank keys never group together."""
    union = a | b
    return len(a & b) / len(union) if union else 0.0


def near_duplicate_groups(queries: Mapping[int, str], threshold: float) -> list[list[int]]:
    """Return connected components (size two or more) of the similarity graph.

    Components rather than pairs: if A resembles B and B resembles C, all three must travel
    together even when A and C fall just under the threshold.
    """
    parent = {query_id: query_id for query_id in queries}

    def find(node: int) -> int:
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    keys = {query_id: token_key(text) for query_id, text in queries.items()}
    for a, b in combinations(sorted(keys), 2):
        if jaccard(keys[a], keys[b]) >= threshold:
            parent[find(b)] = find(a)

    groups: dict[int, list[int]] = {}
    for query_id in sorted(parent):
        groups.setdefault(find(query_id), []).append(query_id)
    return [members for members in groups.values() if len(members) > 1]


In [ ]:
%%writefile shelfrank/data/download.py
"""Fetch pinned files and refuse anything whose content differs from the recorded hash."""

import hashlib
import urllib.request
from collections.abc import Callable
from pathlib import Path

CHUNK = 1 << 20
Opener = Callable[[str, Path], None]


class ChecksumError(RuntimeError):
    """The downloaded bytes are not the bytes the results were computed from."""


def sha256_of(path: Path) -> str:
    """Hash in chunks so the 90 MB product file never has to fit in memory at once."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(CHUNK):
            digest.update(block)
    return digest.hexdigest()


def http_fetch(url: str, target: Path) -> None:
    """Stream ``url`` to ``target``."""
    if not url.startswith("https://"):
        raise ValueError(f"refusing a non-HTTPS download: {url}")
    request = urllib.request.Request(url, headers={"User-Agent": "dataset-fetch/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response, target.open("wb") as out:
        while block := response.read(CHUNK):
            out.write(block)


def ensure_file(url: str, target: Path, sha256: str, fetch: Opener = http_fetch) -> bool:
    """Make ``target`` exist with the expected hash; return True if a download happened.

    The download goes to a temporary name first, so an interrupted run cannot leave a truncated
    file that a later run would mistake for the real one.
    """
    if target.exists() and sha256_of(target) == sha256:
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".part")
    try:
        fetch(url, partial)
        actual = sha256_of(partial)
        if actual != sha256:
            raise ChecksumError(f"{target.name}: expected {sha256}, got {actual}")
        partial.replace(target)
    finally:
        partial.unlink(missing_ok=True)
    return True


In [ ]:
%%writefile shelfrank/qu/seed_selection.py
"""Choose ESCI queries to label. Runs inside the Colab notebook, tested here.

Only pyarrow and the standard library are used, so the notebook needs no installation.
"""

import hashlib
import random
import re
from collections import Counter, defaultdict
from collections.abc import Collection, Iterable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from shelfrank.text.near_duplicates import jaccard, leakage_keys
from shelfrank.text.normalize import tokenize

_ASIN = re.compile(r"^b0[a-z0-9]{8}$")
NO_CLASS = "__no_class__"


@dataclass
class QueryEvidence:
    """How many of a query's Exact products matched each class."""

    text: str
    products: list[str] = field(default_factory=list)
    votes: Counter = field(default_factory=Counter)


def is_usable(query: str) -> bool:
    """Keep plain-text queries a shopper could type; drop product codes and other scripts."""
    words = query.split()
    letters = sum(char.isalpha() for char in query)
    if not query.isascii() or _ASIN.match(query.lower()):
        return False
    return 1 <= len(words) <= 10 and letters >= 3


def term_index(class_terms: Mapping[str, Iterable[str]]) -> dict[tuple[str, ...], str]:
    """Invert class terms for lookup."""
    return {tuple(term.split()): label for label, terms in class_terms.items() for term in terms}


def match_class(title: str, index: Mapping[tuple[str, ...], str], max_n: int = 4) -> str | None:
    """Class of the longest term in a title, earliest on a tie.

    Longest first, so "desk lamp" is read as a lamp and not as a desk. Earliest next, because
    marketplace titles put the product type before the search keywords.
    """
    tokens = tokenize(title)
    for size in range(min(max_n, len(tokens)), 0, -1):
        for start in range(len(tokens) - size + 1):
            label = index.get(tuple(tokens[start : start + size]))
            if label is not None:
                return label
    return None


def read_exact_pairs(examples: Path, locale: str, split: str) -> dict[str, QueryEvidence]:
    """Queries with their Exact products. The filter is pushed into the parquet scan."""
    import pyarrow.dataset as ds

    wanted = (
        (ds.field("product_locale") == locale)
        & (ds.field("esci_label") == "E")
        & (ds.field("split") == split)
    )
    queries: dict[str, QueryEvidence] = {}
    scanner = ds.dataset(examples).scanner(
        columns=["query_id", "query", "product_id"], filter=wanted
    )
    for batch in scanner.to_batches():
        columns = [batch.column(i).to_pylist() for i in range(3)]
        for query_id, text, product_id in zip(*columns, strict=True):
            evidence = queries.setdefault(str(query_id), QueryEvidence(str(text).strip()))
            evidence.products.append(product_id)
    return queries


def add_votes(
    queries: Mapping[str, QueryEvidence],
    products: Path,
    locale: str,
    index: Mapping[tuple[str, ...], str],
    batch_size: int = 20_000,
) -> int:
    """Stream product titles and count class matches per query; return the titles read.

    Batches are dropped after use and only the two needed columns are read, so memory stays flat
    however large the products file is.
    """
    import pyarrow.dataset as ds

    wanted: dict[str, list[str]] = defaultdict(list)
    for query_id, evidence in queries.items():
        for product_id in evidence.products:
            wanted[product_id].append(query_id)
    read = 0
    scanner = ds.dataset(products).scanner(
        columns=["product_id", "product_title"],
        filter=ds.field("product_locale") == locale,
        batch_size=batch_size,
        batch_readahead=2,
        fragment_readahead=1,
    )
    for batch in scanner.to_batches():
        ids, titles = batch.column(0).to_pylist(), batch.column(1).to_pylist()
        for product_id, title in zip(ids, titles, strict=True):
            if product_id in wanted and title:
                read += 1
                label = match_class(title, index)
                if label is not None:
                    for query_id in wanted[product_id]:
                        queries[query_id].votes[label] += 1
    return read


def assign_pools(
    queries: Mapping[str, QueryEvidence], min_votes: int, min_share: float
) -> dict[str, list[str]]:
    """Group query ids by their majority class, or by having no class evidence at all.

    Queries with weak evidence join neither pool. The teacher decides every label, so this only has
    to raise the share of home-goods queries in the sample.
    """
    pools: dict[str, list[str]] = defaultdict(list)
    for query_id, evidence in sorted(queries.items()):
        if not evidence.votes:
            if len(evidence.products) >= min_votes:
                pools[NO_CLASS].append(query_id)
            continue
        label, count = min(evidence.votes.items(), key=lambda item: (-item[1], item[0]))
        if count >= min_votes and count / len(evidence.products) >= min_share:
            pools[label].append(query_id)
    return dict(pools)


def spread(pools: Mapping[str, Sequence[str]], total: int, rng: random.Random) -> list[str]:
    """Take one query per class in turn until ``total``, so rare classes are represented."""
    shuffled = {label: rng.sample(list(ids), len(ids)) for label, ids in sorted(pools.items())}
    chosen: list[str] = []
    while len(chosen) < total and any(shuffled.values()):
        for label in sorted(shuffled):
            if shuffled[label] and len(chosen) < total:
                chosen.append(shuffled[label].pop())
    return chosen


def resembles(query: str, keys: Collection[frozenset[str]], threshold: float) -> bool:
    """Whether a query is the same as, or nearly the same as, any protected query."""
    return any(jaccard(own, key) >= threshold for own in leakage_keys(query) for key in keys)


def filter_queries(
    queries: Mapping[str, QueryEvidence], protected: Iterable[str], threshold: float
) -> tuple[dict[str, QueryEvidence], Counter]:
    """Drop unusable queries, repeats, and anything that resembles a protected query."""
    by_token: dict[str, set[frozenset[str]]] = defaultdict(set)
    for text in protected:
        for key in leakage_keys(text):
            for token in key:
                by_token[token].add(key)
    kept: dict[str, QueryEvidence] = {}
    seen: set[frozenset[str]] = set()
    dropped: Counter = Counter()
    for query_id, evidence in sorted(queries.items()):
        keys = leakage_keys(evidence.text)
        tokens = frozenset().union(*keys)
        nearby = set().union(*(by_token[t] for t in tokens if t in by_token))
        if not keys or not is_usable(evidence.text):
            dropped["unusable"] += 1
        elif seen.intersection(keys):
            dropped["repeat"] += 1
        elif resembles(evidence.text, nearby, threshold):
            dropped["resembles a protected query"] += 1
        else:
            seen.update(keys)
            kept[query_id] = evidence
    return kept, dropped


def select(
    pools: Mapping[str, Sequence[str]],
    n_seed: int,
    n_dev: int,
    home_share: float,
    seed: int,
) -> tuple[list[str], list[str]]:
    """Draw disjoint seed and prompt-development samples from the same mixture."""
    rng = random.Random(seed)
    total = n_seed + n_dev
    n_home = round(total * home_share)
    home = spread({k: v for k, v in pools.items() if k != NO_CLASS}, n_home, rng)
    outside = list(pools.get(NO_CLASS, []))
    chosen = home + rng.sample(outside, min(total - n_home, len(outside)))
    if len(chosen) < total:
        raise ValueError(f"only {len(chosen)} of {total} queries available: loosen the settings")
    rng.shuffle(chosen)
    return chosen[n_dev:], chosen[:n_dev]


def write_outputs(
    directory: Path, seed_texts: Sequence[str], dev_texts: Sequence[str]
) -> dict[str, Any]:
    """Write one query per line and return the hashes for the manifest."""
    directory.mkdir(parents=True, exist_ok=True)
    out = {}
    for name, texts in (("esci_seed_queries.txt", seed_texts), ("esci_prompt_dev.txt", dev_texts)):
        body = "".join(" ".join(text.split()) + "\n" for text in texts).encode("utf-8")
        (directory / name).write_bytes(body)
        out[name] = {"lines": len(texts), "sha256": hashlib.sha256(body).hexdigest()}
    return out


In [ ]:
%%writefile data/class_terms.json
{
 "Accent Chairs": [
  "accent chair",
  "armchair",
  "club chair",
  "inch wide armchair",
  "leather armchair",
  "polyester armchair",
  "velvet armchair",
  "wide armchair",
  "wingback chair"
 ],
 "Area Rugs": [
  "area rug",
  "beige area rug",
  "blue area rug",
  "gray area rug",
  "gray rug",
  "outdoor area rug",
  "plum area rug",
  "rug"
 ],
 "Beds": [
  "bed",
  "platform bed",
  "profile platform bed",
  "profile standard bed",
  "sleigh bed",
  "standard bed",
  "storage platform bed",
  "upholstered platform bed",
  "upholstered standard bed"
 ],
 "Coffee & Cocktail Tables": [
  "abstract coffee table",
  "block coffee table",
  "cocktail table",
  "coffee table",
  "coffee table storage",
  "drum coffee table",
  "leg coffee table",
  "sled coffee table",
  "wood coffee table"
 ],
 "Floor & Wall Tile": [
  "ceramic subway tile",
  "field tile",
  "floor tile",
  "mosaic tile",
  "porcelain field tile",
  "subway tile",
  "tile",
  "wall floor tile",
  "wall tile"
 ],
 "Dining Chairs": [
  "arm chair",
  "chair gray",
  "dining chair",
  "parson chair",
  "side chair",
  "upholstered dining chair",
  "upholstered parson chair",
  "upholstered side chair"
 ],
 "Bar Stools": [
  "bar counter stool",
  "bar stool",
  "counter bar stool",
  "counter stool",
  "height bar stool",
  "inch bar stool",
  "inch counter stool",
  "stool"
 ],
 "Desks": [
  "desk",
  "desk hutch",
  "executive desk",
  "floating desk hutch",
  "shape executive desk",
  "solid wood desk",
  "standing desk",
  "wood desk",
  "writing desk"
 ],
 "End Tables": [
  "drawer end table",
  "drum end table",
  "end table",
  "end table storage",
  "leg end table",
  "nesting table",
  "top end table",
  "wood end table"
 ],
 "Accent Pillows": [
  "accent pillow",
  "cover insert",
  "lumbar pillow",
  "pillow",
  "pillow cover",
  "pillow cover insert",
  "throw pillow",
  "throw pillow cover"
 ],
 "Dining Tables": [
  "dining table",
  "extendable dining table",
  "height dining table",
  "inch dining table",
  "pedestal dining table",
  "pub table",
  "trestle dining table",
  "wood dining table"
 ],
 "Wall Art": [
  "art print",
  "canvas",
  "graphic art print",
  "painting print",
  "print canvas",
  "print wrapped canvas",
  "wall art",
  "wrapped canvas"
 ],
 "Bedding Sets": [
  "comforter",
  "comforter set",
  "cover set",
  "duvet cover set",
  "piece comforter set",
  "quilt set",
  "reversible comforter set",
  "single comforter"
 ],
 "Dressers & Chests": [
  "chest",
  "combo dresser",
  "double dresser",
  "drawer chest",
  "drawer double dresser",
  "drawer dresser",
  "drawer standard dresser",
  "dresser",
  "standard dresser"
 ],
 "Curtains & Drapes": [
  "curtain",
  "curtain drape",
  "curtain panel",
  "drape",
  "grommet curtain panel",
  "inch multicolor",
  "pocket curtain panel",
  "single curtain panel"
 ],
 "Wall Décor": [
  "anchor wall decor",
  "decor",
  "decor set",
  "metal wall decor",
  "sign wall decor",
  "wall decor",
  "wall decor set",
  "window wall decor"
 ],
 "Sofa & Console Tables": [
  "console table",
  "inch console table",
  "sofa table",
  "wood console table",
  "wooden console table"
 ],
 "Vanities": [
  "bathroom vanity",
  "bathroom vanity set",
  "double bathroom vanity",
  "set mirror",
  "single bathroom vanity",
  "single vanity set",
  "vanity",
  "vanity set",
  "vanity set mirror"
 ],
 "Office Chairs": [
  "conference chair",
  "drafting chair",
  "ergonomic executive chair",
  "ergonomic task chair",
  "executive chair",
  "mesh task chair",
  "office chair",
  "task chair"
 ],
 "Recliners": [
  "assist standard recliner",
  "manual standard recliner",
  "massager",
  "recliner",
  "recliner massager",
  "standard recliner",
  "standard recliner massager"
 ],
 "Sofas": [
  "arm loveseat",
  "arm reclining loveseat",
  "arm sleeper",
  "arm sofa",
  "loveseat",
  "reclining loveseat",
  "reversible cushion",
  "sofa",
  "sofa bed"
 ],
 "Cabinet and Drawer Pulls": [
  "arch pull",
  "bar pull",
  "bar pull multipack",
  "cabinet pull",
  "center arch pull",
  "center bar pull",
  "drawer pull",
  "finger pull",
  "pull multipack"
 ],
 "Outdoor Conversation Sets": [
  "group cushion",
  "outdoor conversation set",
  "seating group",
  "seating group cushion"
 ],
 "TV Stands & Entertainment Centers": [
  "electric fireplace included",
  "entertainment center",
  "fireplace included",
  "tv stand"
 ],
 "Wall & Accent Mirrors": [
  "accent mirror",
  "beveled accent mirror",
  "distressed accent mirror",
  "dresser mirror",
  "full length mirror",
  "length mirror",
  "vanity mirror",
  "wall mirror"
 ],
 "Sectionals": [
  "chaise",
  "chaise ottoman",
  "corner sectional",
  "corner sectional ottoman",
  "sectional",
  "sectional ottoman",
  "sofa chaise",
  "sofa chaise ottoman"
 ],
 "Chandeliers": [
  "chandelier",
  "classic traditional chandelier",
  "drum chandelier",
  "geometric chandelier",
  "statement geometric chandelier",
  "style drum chandelier",
  "tiered chandelier",
  "traditional chandelier"
 ],
 "Nightstands": [
  "drawer nightstand",
  "light brown",
  "light gray",
  "nightstand",
  "nightstand light brown",
  "nightstand light gray",
  "solid wood nightstand",
  "wood nightstand"
 ],
 "Kitchen Mats": [
  "anti fatigue mat",
  "comfort kitchen mat",
  "fatigue kitchen mat",
  "fatigue mat",
  "fatigue mat set",
  "kitchen mat",
  "mat set"
 ],
 "Decorative Objects": [
  "decorative object",
  "figurine",
  "figurine set",
  "pineapple sculpture",
  "rooster figurine",
  "sculpture"
 ],
 "Patio Sofas": [
  "daybed cushion",
  "loveseat cushion",
  "patio sectional cushion",
  "patio sofa",
  "patio sofa cushion",
  "sectional cushion",
  "sofa cushion"
 ],
 "Bathroom Sink Faucets": [
  "bathroom faucet",
  "bathroom sink faucet",
  "diamond seal technology",
  "drain assembly",
  "faucet drain assembly",
  "hole bathroom faucet",
  "sink bathroom faucet",
  "widespread bathroom faucet"
 ],
 "Accent Chests / Cabinets": [
  "accent cabinet",
  "accent chest",
  "cabinet",
  "door accent cabinet",
  "drawer accent chest"
 ],
 "Table Lamps": [
  "bedside table lamp",
  "inch standard lamp",
  "inch table lamp",
  "lamp",
  "lamp set",
  "standard lamp",
  "table lamp",
  "table lamp set"
 ],
 "Doormats": [
  "door mat",
  "doormat",
  "indoor door mat",
  "outdoor door mat"
 ],
 "Furniture Cushions": [
  "back cushion",
  "bench cushion",
  "dining chair cushion",
  "furniture cushion",
  "lounge chair cushion",
  "outdoor bench cushion",
  "outdoor seat cushion",
  "seat back cushion",
  "seat cushion"
 ],
 "Ottomans": [
  "cocktail ottoman",
  "ottoman",
  "pouf",
  "pouf ottoman",
  "rectangle standard ottoman",
  "square standard ottoman",
  "square storage ottoman",
  "standard ottoman",
  "storage ottoman"
 ],
 "Kids Beds": [
  "bunk bed",
  "bunk bed trundle",
  "full bunk bed",
  "kid bed",
  "low loft bed",
  "standard bunk bed",
  "toddler bed",
  "twin bunk bed"
 ],
 "Wall Clocks": [
  "clock",
  "inch wall clock",
  "wall clock"
 ],
 "Wall Mounted Shelves": [
  "accent shelf",
  "floating shelf",
  "shelf set",
  "tiered shelf",
  "wall mounted shelf",
  "wall shelf",
  "wall shelf set",
  "wood floating shelf"
 ],
 "Bookcases": [
  "bookcase",
  "etagere bookcase",
  "geometric bookcase",
  "ladder bookcase",
  "metal etagere bookcase",
  "standard bookcase",
  "steel etagere bookcase"
 ],
 "Garden Statues": [
  "book statue",
  "garden gnome statue",
  "garden statue",
  "girl statue",
  "gnome statue",
  "reading book statue",
  "statue",
  "statue set"
 ],
 "Kitchen Islands": [
  "block top",
  "butcher block top",
  "island",
  "island set",
  "kitchen cart",
  "kitchen island",
  "kitchen island set",
  "wood top"
 ],
 "Bathroom Storage": [
  "bathroom cabinet",
  "bathroom shelf",
  "bathroom storage",
  "mounted bathroom cabinet",
  "mounted bathroom shelf",
  "over toilet storage",
  "standing bathroom cabinet",
  "toilet storage"
 ],
 "Shoe Storage": [
  "pair shoe rack",
  "shoe rack",
  "shoe storage",
  "shoe storage bench",
  "shoe storage cabinet",
  "stackable shoe rack"
 ],
 "Sideboards & Buffets": [
  "buffet",
  "buffet table",
  "drawer sideboard",
  "inch wide server",
  "inch wide sideboard",
  "sideboard",
  "wide server",
  "wide sideboard"
 ],
 "Planters": [
  "ceramic statue planter",
  "hanging planter",
  "planter",
  "planter set",
  "plastic pot planter",
  "pot planter",
  "pot planter set",
  "resin pot planter",
  "statue planter"
 ],
 "Sheets And Sheet Sets": [
  "body pillow case",
  "body pillowcase",
  "pillow case",
  "pillowcase",
  "sheet set"
 ],
 "Dining Table Sets": [
  "breakfast dining set",
  "dining set",
  "dining table set",
  "height dining set",
  "piece dining set",
  "pub table set",
  "wood dining set"
 ],
 "Faux Florals": [
  "arrangement glass vase",
  "arrangement planter",
  "arrangement pot",
  "arrangement vase",
  "faux floral",
  "floral arrangement pot",
  "floral arrangement vase",
  "glass vase",
  "vase"
 ],
 "Kids Chairs": [
  "kid chair",
  "kid club chair",
  "kid desk",
  "kid sleeper",
  "kid sofa"
 ],
 "Fountains": [
  "fountain",
  "fountain led light",
  "fountain light",
  "led light",
  "resin fountain light",
  "water fountain",
  "water fountain light"
 ],
 "Vanity Lighting": [
  "bath bar",
  "bath sconce",
  "bath vanity light",
  "chrome vanity light",
  "dimmable vanity light",
  "light bath bar",
  "light vanity light",
  "vanity light",
  "vanity lighting"
 ],
 "Benches": [
  "bench",
  "storage bench",
  "top storage bench",
  "upholstered bench",
  "upholstered storage bench",
  "upholstered wood bench",
  "wood bench"
 ],
 "Office Storage Cabinets": [
  "door storage cabinet",
  "drawer storage cabinet",
  "filing storage cabinet",
  "office storage cabinet",
  "shelf storage cabinet"
 ],
 "Canisters & Jars": [
  "canister",
  "canister set",
  "coffee jar",
  "jar",
  "jar set",
  "kitchen canister",
  "kitchen canister set",
  "sugar jar",
  "sugar jar set"
 ],
 "Bars & Bar Sets": [
  "bar cabinet",
  "bar set",
  "bar wine storage",
  "floor wine cabinet",
  "wine bar",
  "wine cabinet",
  "wine storage"
 ],
 "Headboards": [
  "frame headboard",
  "headboard",
  "open frame headboard",
  "panel headboard",
  "upholstered panel headboard",
  "upholstered wingback headboard",
  "wingback headboard"
 ],
 "Living Room Table Sets": [
  "coffee table set",
  "living room table set",
  "piece nesting table",
  "table set"
 ],
 "Light Bulbs": [
  "candelabra base",
  "led light bulb",
  "light bulb",
  "medium standard base",
  "standard base"
 ],
 "Wall Plates": [
  "outlet wall plate",
  "switch wall plate",
  "wall plate"
 ],
 "Bath Accessories": [
  "accessory set",
  "bath accessory",
  "bathroom accessory set",
  "box cover",
  "tissue box cover"
 ],
 "Serving Dishes & Platters": [
  "condiment server set",
  "melamine platter",
  "platter",
  "serving dish"
 ],
 "Patio Lounge Chairs": [
  "patio chair",
  "patio chair cushion",
  "patio lounge chair"
 ],
 "Indoor Chaise Lounges": [
  "arm chaise lounge",
  "armless chaise lounge",
  "chaise lounge",
  "indoor chaise lounge",
  "lounge"
 ],
 "Patio Dining Sets": [
  "bistro set",
  "bistro set cushion",
  "patio dining set",
  "piece bistro set"
 ],
 "Flush Mount Lighting": [
  "flush mount",
  "flush mount lighting",
  "inch flush mount",
  "semi flush mount"
 ],
 "Licensed Products": [
  "licensed product"
 ],
 "Daybeds & Guest Beds": [
  "daybed",
  "daybed bed",
  "daybed trundle",
  "guest bed",
  "twin daybed",
  "twin daybed trundle"
 ],
 "Cabinet and Drawer Knobs": [
  "cabinet knob",
  "diameter mushroom knob",
  "diameter round knob",
  "drawer knob",
  "knob",
  "knob multipack",
  "length square knob",
  "mushroom knob",
  "round knob",
  "square knob"
 ],
 "Hampers & Baskets": [
  "hamper",
  "laundry basket",
  "laundry hamper"
 ],
 "Shower Curtain Hooks and Accessories": [
  "curtain hook",
  "shower curtain hook",
  "shower hook"
 ],
 "Curtain Hardware & Accessories": [
  "curtain hardware",
  "curtain rod",
  "curtain tension rod",
  "double curtain rod",
  "single curtain rod",
  "tension rod"
 ],
 "Plant & Telephone Tables": [
  "plant stand",
  "plant table",
  "telephone table",
  "tiered plant stand"
 ],
 "Molding & Millwork": [
  "crown molding",
  "millwork",
  "molding",
  "panel molding",
  "wainscot wall panel",
  "wall panel"
 ],
 "Bed Frames": [
  "bed frame",
  "inch bed frame",
  "inch platform bed",
  "steel bed frame",
  "steel platform bed"
 ],
 "Outdoor Wall Decor": [
  "outdoor wall decor",
  "welcome sign"
 ],
 "Kitchen Sinks": [
  "apron kitchen sink",
  "basket strainer",
  "farmhouse kitchen sink",
  "grid basket strainer",
  "grid drain assembly",
  "kitchen sink"
 ],
 "Shower & Tub Doors": [
  "clearmax technology",
  "door clearmax technology",
  "fixed glass panel",
  "frameless shower door",
  "frameless tub door",
  "glass panel",
  "shower door",
  "tub door"
 ],
 "Slow Cookers": [
  "cooker",
  "pressure cooker",
  "slow cooker",
  "steel slow cooker"
 ],
 "Wall Sconces": [
  "armed sconce",
  "dimmable wall sconce",
  "light wall sconce",
  "wall sconce"
 ],
 "Coat Racks and Hooks": [
  "coat rack",
  "mounted coat rack",
  "wall hook"
 ],
 "Shower Faucets & Systems": [
  "faucet rough valve",
  "rough valve",
  "shower faucet",
  "system rough valve"
 ],
 "Kids Wall Décor": [
  "canvas art",
  "framed art",
  "kid wall decor",
  "paper print"
 ],
 "Projector Screens": [
  "electric projection screen",
  "mounted projector screen",
  "portable projection screen",
  "projection screen",
  "projector screen"
 ],
 "Outdoor Wall Lights": [
  "barn light",
  "bulkhead light",
  "outdoor armed sconce",
  "outdoor barn light",
  "outdoor bulkhead light",
  "outdoor wall lantern",
  "outdoor wall light",
  "wall lantern"
 ],
 "Boxes, Bins, Baskets, & Buckets": [
  "basket",
  "basket set",
  "bin",
  "bucket",
  "metal wire basket",
  "wire basket",
  "wire basket set"
 ],
 "Kids Dressers & Chests": [
  "kid dresser"
 ],
 "Shower Curtains": [
  "curtain liner",
  "shower curtain",
  "shower curtain liner",
  "single shower curtain"
 ],
 "Crib Bedding Sets": [
  "bedding set",
  "crib bedding set"
 ],
 "Towel & Robe Hooks": [
  "mounted robe hook",
  "mounted towel hook",
  "robe hook",
  "towel hook"
 ],
 "Living Room Sets": [
  "living room set",
  "room set"
 ],
 "Fixture Parts And Components": [
  "fixture part",
  "lever handle",
  "trim lever handle"
 ],
 "Toilet Paper Holders": [
  "paper holder",
  "toilet paper holder"
 ],
 "Shelving & Racks": [
  "rack",
  "shelving",
  "shelving unit",
  "storage rack",
  "storage shelf"
 ],
 "Furniture Covers": [
  "furniture cover",
  "furniture cover set",
  "set cover",
  "year warranty"
 ],
 "Plates & Saucers": [
  "dessert plate",
  "dinner plate",
  "inch salad plate",
  "melamine salad plate",
  "salad plate",
  "saucer"
 ],
 "Foam and Latex Mattresses": [
  "foam mattress",
  "latex mattress",
  "memory foam mattress"
 ],
 "Patio Rockers & Gliders": [
  "glider",
  "patio rocker",
  "porch rocking chair",
  "rocking chair cushion"
 ],
 "Kitchen Faucets": [
  "handle kitchen faucet",
  "hot water dispenser",
  "kitchen faucet",
  "kitchen sink faucet",
  "water dispenser"
 ],
 "Deck Boxes": [
  "deck box",
  "plasic deck box",
  "plastic deck box",
  "resin deck box"
 ],
 "Vanity Bases": [
  "base only",
  "vanity base",
  "vanity base only"
 ],
 "Carts & Stands": [
  "cart",
  "mobile printer stand",
  "printer stand",
  "utility cart"
 ],
 "Mattress Toppers and Pads": [
  "foam mattress topper",
  "mattress pad",
  "mattress topper",
  "polyester mattress pad",
  "topper"
 ],
 "Faux Plants and Trees": [
  "faux plant",
  "plant planter",
  "plant pot",
  "succulent pot",
  "tree"
 ],
 "Garden Accents": [
  "fairy garden",
  "garden accent"
 ],
 "Holiday Accents & Decor": [
  "holiday accent"
 ],
 "Bath Rugs & Mats": [
  "bath mat",
  "bath rug",
  "bath rug set",
  "foam bath rug",
  "mat",
  "slip bath rug"
 ],
 "Greenhouses": [
  "greenhouse"
 ],
 "Landscape Lighting": [
  "landscape lighting",
  "led pathway light",
  "light pack",
  "pathway light",
  "pathway light pack"
 ],
 "Patio Tables": [
  "patio table",
  "side table"
 ],
 "Trash Cans & Recycling": [
  "gallon waste basket",
  "trash can",
  "waste basket"
 ],
 "Patio Chaise Lounges": [
  "patio chaise lounge",
  "reclining chaise lounge"
 ],
 "Folding Tables": [
  "adjustable folding table",
  "folding table",
  "half folding table",
  "portable folding table"
 ],
 "Sheds": [
  "lean storage shed",
  "metal storage shed",
  "plastic storage shed",
  "shed",
  "storage shed",
  "tool shed"
 ],
 "Food Storage & Dispensers": [
  "bread box",
  "dispenser",
  "food storage",
  "food storage container",
  "food storage set",
  "storage container",
  "storage set"
 ],
 "Clothing & Garment Racks": [
  "clothing rack",
  "garment rack"
 ],
 "Chairmats": [
  "chair mat",
  "chairmat",
  "edge chair mat",
  "rectangular chair mat"
 ],
 "Patio Bar Stools": [
  "patio bar stool"
 ],
 "Range Hoods": [
  "cabinet range hood",
  "hood stainless steel",
  "mount range hood",
  "range hood",
  "stainless steel"
 ],
 "Picture Frames": [
  "picture frame"
 ],
 "Bread & Loaf Pans": [
  "bread pan",
  "loaf pan",
  "stick loaf pan"
 ],
 "Futons": [
  "convertible sofa",
  "futon"
 ],
 "Under Cabinet Lighting": [
  "cabinet puck light",
  "cabinet strip light",
  "puck light",
  "strip light",
  "under cabinet lighting"
 ],
 "Lighting Shades": [
  "lamp shade screw",
  "lighting shade",
  "pendant shade screw",
  "shade screw"
 ],
 "Candle Holders": [
  "candle holder",
  "iron wall sconce"
 ],
 "Kids Desks": [
  "kid study desk",
  "study desk"
 ],
 "Ceiling Fans": [
  "ceiling fan",
  "kit included",
  "light kit included",
  "standard ceiling fan"
 ],
 "Shower Heads": [
  "dual shower head",
  "handheld shower head",
  "shower head"
 ],
 "Dining Bowls": [
  "cereal bowl",
  "dessert bowl",
  "dining bowl"
 ],
 "Armoires & Wardrobes": [
  "armoire",
  "wardrobe",
  "wardrobe armoire"
 ],
 "Bike And Sport Racks": [
  "bike rack",
  "freestanding sport rack",
  "mounted sport rack",
  "sport rack"
 ],
 "Dining Linens": [
  "dining linen",
  "table runner",
  "tablecloth",
  "vinyl tablecloth"
 ],
 "Outdoor Fireplaces": [
  "fire pit",
  "fire pit table",
  "outdoor fireplace",
  "pit table"
 ],
 "Planter Accessories": [
  "planter accessory"
 ],
 "Flatware & Silverware Serving Pieces": [
  "cake knife",
  "cake serving set",
  "flatware piece",
  "serving set",
  "silverware serving piece"
 ],
 "Teen Desk Chairs": [
  "teen desk chair"
 ],
 "Adjustable Beds": [
  "adjustable bed",
  "bed wireless remote",
  "wireless remote"
 ],
 "Office Sets": [
  "office set"
 ],
 "Patio Umbrella Stands & Bases": [
  "patio umbrella stand",
  "standing umbrella base",
  "umbrella base"
 ],
 "Strainers, Colanders, & Salad Spinners": [
  "colander set",
  "colander spinner",
  "salad spinner",
  "stainless steel colander",
  "steel colander",
  "steel colander set",
  "strainer set",
  "strainer spinner"
 ],
 "Safes": [
  "electronic lock",
  "key lock",
  "safe"
 ],
 "Closet Storage & Organization": [
  "closet rod",
  "closet storage",
  "closet system"
 ],
 "Ornaments & Tree Décor": [
  "figurine ornament",
  "hanging figurine ornament",
  "holiday shaped ornament",
  "ornament",
  "ornament decor",
  "shaped ornament",
  "tree decor"
 ],
 "Reception Sofas & Loveseats": [
  "reception sofa"
 ],
 "Wall Stickers": [
  "decal",
  "wall decal",
  "wall sticker"
 ],
 "Door Levers": [
  "door lever",
  "passage door lever"
 ],
 "Innerspring Mattresses": [
  "firm innerspring mattress",
  "innerspring mattress",
  "medium innerspring mattress",
  "pillow top mattress",
  "plush innerspring mattress",
  "top mattress"
 ],
 "Massage Chairs": [
  "heated massage chair",
  "massage chair",
  "reclining massage chair"
 ],
 "Mugs & Teacups": [
  "coffee mug",
  "mug",
  "teacup"
 ],
 "Barn Door Hardware": [
  "barn door hardware",
  "door hardware kit",
  "hardware kit"
 ],
 "Christmas Trees": [
  "artificial christmas tree",
  "christmas tree",
  "clear white light",
  "white light"
 ],
 "Pantry Cabinets": [
  "inch kitchen pantry",
  "kitchen pantry",
  "pantry",
  "pantry cabinet"
 ],
 "Signs & Sign Holders": [
  "inch wide",
  "name room sign",
  "room sign",
  "sign holder"
 ],
 "Teen Lounge Furniture": [
  "teen lounge furniture"
 ],
 "Shower & Tub Accessories": [
  "caddy",
  "shower accessory",
  "shower caddy",
  "tub accessory"
 ],
 "Wall Paneling": [
  "paneling white",
  "wall paneling",
  "wood wall paneling"
 ],
 "Pergolas": [
  "pergola",
  "pergola canopy",
  "solid wood pergola",
  "vinyl pergola",
  "wood pergola"
 ],
 "Makeup Vanities": [
  "makeup vanity"
 ],
 "Poker & Casino Tables": [
  "card table",
  "casino table",
  "poker table"
 ],
 "Tubs And Whirlpools": [
  "acrylic bathtub",
  "bathtub",
  "bathtub faucet",
  "freestanding soaking bathtub",
  "soaking acrylic bathtub",
  "soaking bathtub",
  "tub",
  "whirlpool"
 ],
 "Bathroom Sinks": [
  "bathroom sink",
  "bathroom sink overflow",
  "sink overflow",
  "vessel bathroom sink"
 ],
 "Specialty Serving": [
  "cake stand",
  "specialty serving"
 ],
 "Teen Beds": [
  "teen bed"
 ],
 "Weather Instruments": [
  "clock thermometer",
  "thermometer",
  "weather instrument"
 ],
 "Reception Benches": [
  "reception bench"
 ],
 "Refrigerators": [
  "energy star refrigerator",
  "refrigerator",
  "star refrigerator"
 ],
 "Computer Mounts": [
  "computer mount",
  "desk mount",
  "screen desk mount"
 ],
 "Pet Gates": [
  "gate",
  "pet gate",
  "standing pet gate"
 ],
 "Mailboxes": [
  "brass mail slot",
  "mail slot",
  "mailbox",
  "mounted mailbox",
  "wall mounted mailbox"
 ],
 "Memo Boards": [
  "calendar board",
  "dry erase board",
  "erase board",
  "memo board",
  "mounted calendar board"
 ],
 "Fencing & Accessories": [
  "fencing",
  "privacy screen",
  "vinyl privacy screen"
 ],
 "Outdoor Lanterns & Lamps": [
  "outdoor lantern"
 ],
 "Cooking Utensils": [
  "cooking utensil",
  "utensil set"
 ],
 "Jewelry Armoires": [
  "armoire mirror",
  "jewelry armoire",
  "jewelry armoire mirror"
 ],
 "Jewelry Boxes": [
  "jewelry box",
  "jewelry organizer tray",
  "organizer tray"
 ],
 "Standard Bed Pillows": [
  "body pillow",
  "standard bed pillow",
  "support pillow"
 ],
 "Teen Desks": [
  "teen desk"
 ],
 "Awnings": [
  "awning",
  "patio awning",
  "retractable side awning",
  "side awning",
  "standard patio awning"
 ],
 "Log Storage": [
  "firewood log rack",
  "log rack",
  "log storage"
 ],
 "Stackable Chairs": [
  "stackable chair"
 ],
 "Bed Accessories": [
  "bed accessory",
  "bed riser"
 ],
 "Covers & Carry Bags": [
  "carry bag",
  "cover bag"
 ],
 "Decorative Trays": [
  "accent tray set",
  "decorative tray",
  "serving tray set",
  "tray set"
 ],
 "Interior Doors": [
  "closet door",
  "installation hardware kit",
  "interior door",
  "sliding closet door"
 ],
 "Reception Seating Chairs": [
  "reception seating chair"
 ],
 "Towel Bars, Racks, and Stands": [
  "mounted towel bar",
  "towel bar"
 ],
 "Wine Racks": [
  "bottle glass rack",
  "glass rack",
  "wine glass rack",
  "wine rack"
 ],
 "Desktop Organizers & Desk Pads": [
  "desk pad",
  "desktop organizer",
  "monitor stand"
 ],
 "Game Chairs": [
  "game chair",
  "gaming chair"
 ],
 "Pendant Lights": [
  "pendant",
  "pendant light"
 ],
 "Holiday Lighting": [
  "holiday lighting"
 ],
 "Saunas": [
  "sauna",
  "steam sauna",
  "traditional steam sauna"
 ],
 "Bird Cages": [
  "bird cage"
 ],
 "Rocking Chairs": [
  "rocking chair"
 ],
 "Glassware & Barware": [
  "barware",
  "glassware"
 ],
 "Lighting Accessories": [
  "lighting accessory",
  "power strip"
 ],
 "Office Stools": [
  "office stool"
 ],
 "Storage Drawers": [
  "drawer storage chest",
  "storage chest",
  "storage drawer"
 ],
 "Hybrid Mattresses": [
  "hybrid mattress"
 ],
 "Teen Dressers": [
  "teen dresser"
 ],
 "Vases, Urns, Jars, & Bottles": [
  "urn"
 ],
 "Washer and Dryer Sets": [
  "dryer set",
  "electric dryer",
  "gas dryer",
  "washer set"
 ],
 "Closet Organizer Accessories": [
  "closet organizer accessory",
  "hanging organizer"
 ],
 "Flags": [
  "flag",
  "garden flag",
  "inch garden flag"
 ],
 "Kids Headboards": [
  "kid headboard",
  "twin bookcase headboard"
 ],
 "Lawn Games": [
  "lawn game"
 ],
 "Valances & Kitchen Curtains": [
  "kitchen curtain",
  "valance curtain"
 ],
 "Accent Stools": [
  "accent stool"
 ],
 "Dog and Cat Bowls, Feeders & Accessories": [
  "cat bowl",
  "feeder"
 ],
 "Teens Headboards": [
  "teen headboard"
 ]
}


In [ ]:
from shelfrank.data.download import ensure_file

targets = {"examples": EXAMPLES, "products": PRODUCTS, "wands_queries": WANDS_QUERIES}
for name, spec in SOURCES.items():
    if DOWNLOAD:
        fetched = ensure_file(spec["url"], targets[name], spec["sha256"])
        print(("downloaded" if fetched else "already have"), name, f"{time.time() - started:.0f}s")

In [ ]:
import csv, json
from collections import Counter
from shelfrank.qu import seed_selection as ss

terms = json.loads(Path("data/class_terms.json").read_text(encoding="utf-8"))
index = ss.term_index(terms)
queries = ss.read_exact_pairs(EXAMPLES, SETTINGS["locale"], SETTINGS["split"])
print(f"{len(queries):,} queries with an Exact product, {time.time() - started:.0f}s")

# WANDS queries are read by code only, to remove look-alikes. They are never printed or saved.
with WANDS_QUERIES.open(encoding="utf-8", newline="") as handle:
    protected = [row["query"] for row in csv.DictReader(handle, delimiter="\t")]
kept, dropped = ss.filter_queries(queries, protected, SETTINGS["near_duplicate_jaccard"])
print(f"{len(kept):,} kept, dropped {dict(dropped)}")

titles = ss.add_votes(kept, PRODUCTS, SETTINGS["locale"], index)
pools = ss.assign_pools(kept, SETTINGS["min_votes"], SETTINGS["min_share"])
home = {label: ids for label, ids in pools.items() if label != ss.NO_CLASS}
print(f"{titles:,} titles read, {sum(map(len, home.values())):,} home-goods queries in "
      f"{len(home)} classes, {len(pools.get(ss.NO_CLASS, [])):,} outside, {time.time() - started:.0f}s")

In [ ]:
import platform
import pyarrow

seed_ids, dev_ids = ss.select(
    pools, SETTINGS["n_seed"], SETTINGS["n_dev"], SETTINGS["home_share"], SETTINGS["seed"])
files = ss.write_outputs(
    OUT_DIR, [kept[i].text for i in seed_ids], [kept[i].text for i in dev_ids])
try:
    import resource
    peak_gb = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2)
except ImportError:
    peak_gb = None

label_of = {i: label for label, ids in pools.items() for i in ids}
picked = Counter(label_of[i] for i in seed_ids + dev_ids)
manifest = {
    "settings": SETTINGS,
    "sources": SOURCES,
    "code_sha256": CODE_SHA256,
    "files": files,
    "queries_with_exact": len(queries),
    "dropped": dict(sorted(dropped.items())),
    "titles_read": titles,
    "classes_with_candidates": len(home),
    "home_candidates": sum(map(len, home.values())),
    "outside_candidates": len(pools.get(ss.NO_CLASS, [])),
    "picked_outside": picked.get(ss.NO_CLASS, 0),
    "picked_classes": len(set(picked) - {ss.NO_CLASS}),
    "max_picked_per_class": max((n for k, n in picked.items() if k != ss.NO_CLASS), default=0),
    "peak_memory_gb": peak_gb,
    "seconds": round(time.time() - started),
    "python": platform.python_version(),
    "pyarrow": pyarrow.__version__,
}
(OUT_DIR / "esci_seed_manifest.json").write_text(json.dumps(manifest, indent=1) + "\n", encoding="utf-8")
print(json.dumps({k: v for k, v in manifest.items() if k not in ("sources", "code_sha256")}, indent=1))

In [ ]:
# Colab downloads the three files. Elsewhere, copy them from OUT_DIR.
for path in sorted(OUT_DIR.iterdir()):
    print(path, path.stat().st_size, "bytes")
try:
    from google.colab import files as colab_files
    for path in sorted(OUT_DIR.iterdir()):
        colab_files.download(str(path))
except ImportError:
    pass